# Week 14 — GEE Advanced: Long-term Trends & Climate Resilience
## ARIA v9.5 — Student Exercise Notebook (學生練習版)

**Course (課程):** Remote Sensing and Spatial Information Analysis and Applications
**Theme (主題):** Landsat Multi-Decadal Trend Analysis & Resilience Monitoring
**Study Area (研究區域):**
- **Lab 1, 3, 4:** Hualien / Taroko — 26-year vegetation trends & earthquake resilience
- **Lab 2:** Taoyuan Plateau — 26-year pond (埤塘) disappearance analysis

---

### Learning Objectives (學習目標)
1. Harmonize multi-sensor Landsat archives for consistent long-term analysis
2. Compute pixel-level linear trends to map greening vs browning
3. Use MNDWI to detect water body changes — Taoyuan pond disappearance over 26 years
4. Quantify ecosystem resilience after the 2024 earthquake

### How to Use This Notebook (使用說明)
- Cells marked **COMPLETE** are ready to run — do not modify them.
- Cells with `# TODO:` require you to **fill in the blanks** before running.
- `# HINT:` comments provide guidance for each exercise.
- Run cells **in order** from top to bottom.

### Upgrade from W13 (相較 W13 的升級)
```
W13: Sentinel-2  (6 yrs,  10m) → "What changed after the earthquake?"
W14: Landsat     (26 yrs, 30m) → "What has been changing for TWO DECADES?"
     + Taoyuan pond disappearance analysis (MNDWI)
     + Resilience metrics (recovery rate after disturbance)
```

> **Student Exercise Notebook** — fill in the blanks and run.

---
## S1 — Environment Setup (環境設定) ✅ COMPLETE

Same GEE setup as W13. Replace `'your-project-id'` with your Cloud Project ID.

In [ ]:
# ============================================================
# S1 — Environment Setup — COMPLETE
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import ee, geemap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import pandas as pd
from datetime import datetime, timedelta
import os, platform

# ee.Authenticate()  # Uncomment on first run
ee.Initialize(project='your-project-id')

from matplotlib import font_manager as fm
def setup_chinese_font():
    system = platform.system()
    candidates = {
        'Windows': ['Microsoft JhengHei', 'Microsoft YaHei', 'SimHei'],
        'Darwin':  ['PingFang TC', 'Heiti TC', 'STHeiti'],
        'Linux':   ['Noto Sans CJK TC', 'WenQuanYi Micro Hei',
                    'AR PL UMing TW', 'Noto Sans TC']
    }
    available = {f.name for f in fm.fontManager.ttflist}
    for name in candidates.get(system, candidates['Linux']):
        if name in available:
            plt.rcParams['font.sans-serif'] = [name] + plt.rcParams['font.sans-serif']
            plt.rcParams['axes.unicode_minus'] = False
            return name
    return None
setup_chinese_font()

# --- Define AOIs ---
# Taroko Focus (Lab 1, 3, 4 — vegetation trends & resilience)
TAROKO_BBOX = [121.34526379253053, 24.046021742135874,
               121.85149217685861, 24.35767637905926]
aoi = ee.Geometry.Rectangle(TAROKO_BBOX)

# Taoyuan Plateau — FULL RANGE (D6 水頻率圖)
# 桃園台地完整範圍：西至新豐/湖口海岸，東至龍潭丘陵
TAOYUAN_BBOX = [120.94, 24.83, 121.35, 25.08]
aoi_taoyuan = ee.Geometry.Rectangle(TAOYUAN_BBOX)

# Taoyuan URBANIZATION CORRIDOR (D7 消失偵測聚焦區)
# 中壢、新屋、高鐵桃園站沿線
TAOYUAN_URBAN_BBOX = [121.00, 24.88, 121.28, 25.05]
aoi_taoyuan_urban = ee.Geometry.Rectangle(TAOYUAN_URBAN_BBOX)

point = ee.Geometry.Point([121.5, 24.2])
elev = ee.Image('USGS/SRTMGL1_003').sample(point, 30).first().get('elevation').getInfo()
print(f"  Connectivity OK — Elevation: {elev} m")
print(f"  AOI 1 (Lab 1/3/4): Taroko Focus — {TAROKO_BBOX}")
print(f"  AOI 2 (Lab 2, D6): Taoyuan Full  — {TAOYUAN_BBOX}")
print(f"  AOI 3 (Lab 2, D7): Taoyuan Urban — {TAOYUAN_URBAN_BBOX}")

---
## S2 — Landsat Band Harmonization (Landsat 波段統一) ✏️ EXERCISE

Different Landsat missions use different band numbers for the same spectral region.
Your task is to complete the **band renaming functions** so we can merge all four
missions into a single harmonized collection.

| Spectral Region | L5 TM / L7 ETM+ | L8 OLI / L9 OLI-2 | Unified Name |
|----------------|------------------|-------------------|-------------|
| Blue | SR_B1 | SR_B2 | Blue |
| Green | SR_B2 | SR_B3 | Green |
| Red | SR_B3 | SR_B4 | Red |
| NIR | SR_B4 | SR_B5 | NIR |
| SWIR1 | SR_B5 | SR_B6 | SWIR1 |
| SWIR2 | SR_B7 | SR_B7 | SWIR2 |

> **HINT:** `image.select(['old_name1', 'old_name2', ...], ['new_name1', 'new_name2', ...])`

In [ ]:
# ============================================================
# S2 — Landsat Band Harmonization — EXERCISE
# ============================================================

def rename_l57(image):
    """Rename Landsat 5/7 bands to unified names."""
    # TODO: Map L5/L7 bands to unified names
    # L5/L7: SR_B1=Blue, SR_B2=Green, SR_B3=Red, SR_B4=NIR, SR_B5=SWIR1, SR_B7=SWIR2
    return image.select(
        ['SR_B1', 'SR_B2', '____', '____', '____', 'SR_B7', 'QA_PIXEL'],
        ['Blue',  'Green', '____', '____', '____', 'SWIR2', 'QA_PIXEL']
    )

def rename_l89(image):
    """Rename Landsat 8/9 bands to unified names."""
    # TODO: Map L8/L9 bands to unified names
    # L8/L9: SR_B2=Blue, SR_B3=Green, SR_B4=Red, SR_B5=NIR, SR_B6=SWIR1, SR_B7=SWIR2
    return image.select(
        ['SR_B2', 'SR_B3', '____', '____', '____', 'SR_B7', 'QA_PIXEL'],
        ['Blue',  'Green', '____', '____', '____', 'SWIR2', 'QA_PIXEL']
    )

def apply_scale_factors(image):
    """Apply Landsat C2 L2 scale factors: value * 0.0000275 + (-0.2)"""
    optical = image.select(['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']) \
        .multiply(0.0000275).add(-0.2).clamp(0, 1)
    return image.addBands(optical, overwrite=True)

def mask_landsat_clouds(image):
    """Mask clouds using QA_PIXEL bitmask."""
    qa = image.select('QA_PIXEL')
    # TODO: Create cloud and shadow masks using bitwise operations
    # HINT: Bit 3 = cloud, Bit 4 = cloud shadow
    # Pattern: qa.bitwiseAnd(1 << bit_number).eq(0)
    cloud = qa.bitwiseAnd(1 << ____).eq(0)
    shadow = qa.bitwiseAnd(1 << ____).eq(0)
    return image.updateMask(cloud.And(shadow))

# --- Load and merge (COMPLETE — do not modify) ---
DATE_START = '2000-01-01'
DATE_END = '2026-03-31'

l5 = ee.ImageCollection('LANDSAT/LT05/C02/T1_L2').filterDate(DATE_START, '2012-12-31').filterBounds(aoi).map(rename_l57)
l7 = ee.ImageCollection('LANDSAT/LE07/C02/T1_L2').filterDate(DATE_START, DATE_END).filterBounds(aoi).map(rename_l57)
l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2013-01-01', DATE_END).filterBounds(aoi).map(rename_l89)
l9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterDate('2021-01-01', DATE_END).filterBounds(aoi).map(rename_l89)

landsat_all = l5.merge(l7).merge(l8).merge(l9)
landsat_clean = landsat_all.map(mask_landsat_clouds).map(apply_scale_factors)

total = landsat_clean.size().getInfo()
print(f"  Total Landsat images (2000–2026): {total}")
print(f"  L5: {l5.size().getInfo()} | L7: {l7.size().getInfo()} | "
      f"L8: {l8.size().getInfo()} | L9: {l9.size().getInfo()}")

---
## S3 — NDVI & MNDWI Index Calculation (指標計算) ✏️ EXERCISE

Compute two spectral indices from the harmonized Landsat collection:

| Index | Formula | Detects |
|-------|---------|---------|
| **NDVI** | (NIR − Red) / (NIR + Red) | Vegetation health |
| **MNDWI** | (Green − SWIR1) / (Green + SWIR1) | Water bodies |

> **HINT:** Use `normalizedDifference(['band1', 'band2'])` — it computes (b1−b2)/(b1+b2)

In [ ]:
# ============================================================
# S3 — Compute NDVI and MNDWI — EXERCISE
# ============================================================

def add_indices(image):
    # TODO: Compute NDVI = (NIR - Red) / (NIR + Red)
    # HINT: image.normalizedDifference(['NIR', 'Red'])
    ndvi = image.normalizedDifference(['____', '____']).rename('NDVI')

    # TODO: Compute MNDWI = (Green - SWIR1) / (Green + SWIR1)
    mndwi = image.normalizedDifference(['____', '____']).rename('MNDWI')

    return image.addBands([ndvi, mndwi]).copyProperties(image, ['system:time_start'])

landsat_idx = landsat_clean.map(add_indices)
print(f"  Collection with indices: {landsat_idx.size().getInfo()} images")
print("  Bands: NDVI (vegetation) + MNDWI (water)")

---
## S4 — Annual NDVI Time Series (26-Year) ✏️ EXERCISE

Compute **annual median NDVI** for each year from 2000 to 2026.
The function is provided — your job is to create the plot with:
1. Mean, min, max lines (like W13 S4)
2. A linear trend line
3. Event markers for the 2024 earthquake and 2009 Typhoon Morakot

In [ ]:
# ============================================================
# S4 — Annual NDVI Composites — PARTIALLY COMPLETE
# ============================================================

# --- Function is COMPLETE ---
def compute_annual_ndvi(collection, aoi, start_year=2000, end_year=2026):
    results = []
    for year in range(start_year, end_year + 1):
        annual = collection.filterDate(f'{year}-01-01', f'{year}-12-31').select('NDVI')
        n = annual.size().getInfo()
        if n == 0:
            results.append((year, None, None, None, n))
            continue
        median_img = annual.median()
        stats = median_img.reduceRegion(
            reducer=ee.Reducer.mean()
                .combine(ee.Reducer.min(), sharedInputs=True)
                .combine(ee.Reducer.max(), sharedInputs=True),
            geometry=aoi, scale=100, maxPixels=1e8
        ).getInfo()
        v_mean = stats.get('NDVI_mean')
        v_min  = stats.get('NDVI_min')
        v_max  = stats.get('NDVI_max')
        results.append((year, v_mean, v_min, v_max, n))
        if v_mean is not None:
            print(f'  {year}: mean={v_mean:.4f}  min={v_min:.4f}  max={v_max:.4f}  (n={n})')
        else:
            print(f'  {year}: no data (n={n})')
    return results

print('Computing 26-year annual NDVI...')
annual_data = compute_annual_ndvi(landsat_idx, aoi)

years  = [y for y, m, mn, mx, n in annual_data if m is not None]
means  = [m for y, m, mn, mx, n in annual_data if m is not None]
mins   = [mn for y, m, mn, mx, n in annual_data if m is not None]
maxs   = [mx for y, m, mn, mx, n in annual_data if m is not None]

In [ ]:
# ============================================================
# S4 (continued) — Plot 26-Year NDVI — EXERCISE
# ============================================================

fig, ax = plt.subplots(figsize=(14, 6))

# TODO: Add shaded band between min and max
# HINT: ax.fill_between(years, mins, maxs, alpha=0.15, color='green', label='...')
ax.fill_between(years, ____, ____, alpha=0.15, color='green',
                label='Spatial range (min–max)')

# Plot mean line
ax.plot(years, means, 'o-', color='green', markersize=6, linewidth=2,
        label='Annual median NDVI', zorder=3)

# TODO: Add a linear trend line using numpy polyfit
# HINT: z = np.polyfit(years, means, 1); p = np.poly1d(z)
z = np.polyfit(years, means, ____)
p = np.poly1d(z)
ax.plot(years, p(years), ':', color='navy', linewidth=2,
        label=f'Trend: {z[0]:+.5f}/yr')

# TODO: Add earthquake marker (2024) and Morakot marker (2009)
# HINT: ax.axvline(year, color='red', linestyle='--', ...)
ax.axvline(____, color='red', linestyle='--', linewidth=2, label='2024 Hualien EQ')
ax.axvline(____, color='orange', linestyle='--', linewidth=1.5, label='2009 Morakot')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('NDVI', fontsize=12)
ax.set_title('Taroko 26-Year NDVI Time Series (2000–2026)\n'
             '太魯閣 26 年 NDVI 時間序列', fontsize=14)
ax.legend(loc='lower left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\n  26-year trend: {z[0]:+.5f} NDVI/year')
print(f'  Total change: {z[0]*26:+.4f} over 26 years')

### S4 — Reflection (反思)

**Q1:** Is the 26-year NDVI trend positive (greening) or negative (browning)?
What might explain this long-term trend?

> *Your answer:*

**Q2:** Can you spot the effects of Typhoon Morakot (2009) and the 2024 Earthquake
in the time series? How do they compare in terms of NDVI impact?

> *Your answer:*

**Q3:** Compare this 26-year Landsat plot with the 6-year Sentinel-2 plot from W13.
What new information does the longer time series reveal?

> *Your answer:*

---
## S5 — Pixel-Level Trend Map (逐像素趨勢圖) ✏️ EXERCISE

Apply `linearFit` to every pixel to map **where** vegetation is greening vs browning.

> **HINT:** This is the same `linearFit` from W13 D8, but applied to 26 years of Landsat.

In [ ]:
# ============================================================
# S5 — Pixel-Level Trend — EXERCISE
# ============================================================

def add_time_band(image):
    date = ee.Date(image.get('system:time_start'))
    years = date.difference(ee.Date('2000-01-01'), 'year')
    return image.addBands(ee.Image(years).float().rename('year'))

# TODO: Apply time band to the NDVI collection
# HINT: landsat_idx.select('NDVI').map(add_time_band)
ndvi_with_time = landsat_idx.select('____').map(add_time_band)

# TODO: Apply linearFit reducer
# HINT: .select(['year', 'NDVI']).reduce(ee.Reducer.linearFit())
trend = ndvi_with_time.select(['____', '____']).reduce(ee.Reducer.linearFit())
slope = trend.select('scale')

# --- Visualization ---
vis_trend = {
    'min': -0.01, 'max': 0.01,
    'palette': ['d73027', 'fc8d59', 'fee08b', 'ffffbf',
                'd9ef8b', '91cf60', '1a9850']
}

Map5 = geemap.Map(center=[24.20, 121.60], zoom=10)
Map5.addLayer(slope.clip(aoi), vis_trend, 'NDVI Trend (slope/year)')
Map5.addLayer(aoi, {'color': 'yellow'}, 'AOI')
Map5

---
## S6 — Taoyuan Pond Disappearance (桃園埤塘消失偵測) ✏️ EXERCISE

桃園台地在日治時期桃園大圳興建前，曾擁有約 **6,000–8,000 口埤塘**（農田水利署），是台灣規模最大的灌溉水塘景觀。
隨著都市化發展，大量埤塘被填平，目前僅存約 3,000 口。

Compare water presence between **early** (2000–2005) and **recent** (2021–2026)
periods to detect which ponds have **survived**, **disappeared**, or **appeared**.

MNDWI > 0.1 → classified as water

> **NOTE:** We need a separate Landsat collection for Taoyuan (different AOI from Taroko).
> **HINT:** Compute median MNDWI for each period, then threshold at 0.1.

In [ ]:
# ============================================================
# S6 — Taoyuan Pond Disappearance — EXERCISE
#       + True-Color Verification Layers
# ============================================================

# --- Load Landsat for Taoyuan (same functions from S2) ---
l5_ty = ee.ImageCollection('LANDSAT/LT05/C02/T1_L2').filterDate(DATE_START, '2012-12-31').filterBounds(aoi_taoyuan).map(rename_l57)
l7_ty = ee.ImageCollection('LANDSAT/LE07/C02/T1_L2').filterDate(DATE_START, DATE_END).filterBounds(aoi_taoyuan).map(rename_l57)
l8_ty = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2013-01-01', DATE_END).filterBounds(aoi_taoyuan).map(rename_l89)
l9_ty = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterDate('2021-01-01', DATE_END).filterBounds(aoi_taoyuan).map(rename_l89)

landsat_taoyuan = (l5_ty.merge(l7_ty).merge(l8_ty).merge(l9_ty)
                   .map(mask_landsat_clouds)
                   .map(apply_scale_factors)
                   .map(add_indices))

print(f'  Taoyuan Landsat images: {landsat_taoyuan.size().getInfo()}')

# TODO: Compute early period water mask (2000–2005)
# HINT: Filter by date → select MNDWI → median → gt(0.1)
early_water = (landsat_taoyuan
    .filterDate('____', '____')
    .select('MNDWI').median()
    .gt(____).rename('early_water'))

# TODO: Compute recent period water mask (2021–2026)
recent_water = (landsat_taoyuan
    .filterDate('____', '____')
    .select('MNDWI').median()
    .gt(____).rename('recent_water'))

# TODO: Detect changes
# Lost ponds: WAS water before, NOT water now (filled for development)
# New water: NOT water before, IS water now (new retention ponds)
water_loss = early_water.And(recent_water.Not()).selfMask()
water_gain = recent_water.And(early_water.Not()).selfMask()
stable_water = early_water.And(recent_water).selfMask()

# ── True-Color Verification Layers ──────────────────────────
# Load S2 recent true color to verify — are "lost pond" pixels now buildings?
def mask_s2_clouds_ty(image):
    scl = image.select('SCL')
    return image.updateMask(scl.gte(4).And(scl.lte(7)))

s2_taoyuan = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi_taoyuan_urban)
    .filterDate('2024-01-01', '2025-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .map(mask_s2_clouds_ty))

s2_recent_rgb = s2_taoyuan.select(['B4', 'B3', 'B2']).median()

# Early Landsat true color to confirm ponds existed in 2000–2005
early_rgb = (landsat_taoyuan
    .filterDate('2000-01-01', '2005-12-31')
    .select(['Red', 'Green', 'Blue']).median())

# Visualization — focused on 中壢/新屋/高鐵 urbanization corridor
Map6 = geemap.Map(center=[24.96, 121.14], zoom=12)

# Base layers: true-color verification
Map6.addLayer(early_rgb.clip(aoi_taoyuan_urban),
              {'bands': ['Red', 'Green', 'Blue'], 'min': 0, 'max': 0.25},
              'Landsat 早期真彩色 2000–2005 (30m)')
Map6.addLayer(s2_recent_rgb.clip(aoi_taoyuan_urban),
              {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 2500},
              'S2 近期真彩色 2024–2025 (10m)')

# Change detection overlay
Map6.addLayer(stable_water.clip(aoi_taoyuan_urban), {'palette': ['0000FF']},
              'Stable ponds 存活埤塘')
Map6.addLayer(water_gain.clip(aoi_taoyuan_urban), {'palette': ['00FF00']},
              'New water 新增水體')
Map6.addLayer(water_loss.clip(aoi_taoyuan_urban), {'palette': ['FF0000']},
              'Lost ponds 消失埤塘')
Map6.addLayer(aoi_taoyuan_urban, {'color': 'orange'}, 'AOI — 中壢/新屋/高鐵走廊')
Map6.addLayer(aoi_taoyuan, {'color': 'yellow'}, 'AOI — Taoyuan Full')
Map6

# HINT: Toggle layers to verify:
# 1. Open S2 true color → see buildings/roads in recent imagery
# 2. Turn on "Lost ponds" (red) → confirm red pixels match built-up areas
# 3. Switch to Landsat early true color → see dark ponds at the same locations

### S6 — Your Observations (你的觀察)

**Q1:** Where are the main areas of pond loss (red)? Do they correspond to
known urban development zones in Taoyuan (e.g., near 中壢、桃園市區)?

> *Your answer:*

**Q2:** Are there any areas of new water (green)? What might these represent —
new retention ponds for flood control, or construction excavation sites?

> *Your answer:*

**Q3:** 桃園埤塘曾是重要的灌溉系統和防洪設施。從你的分析結果來看，
埤塘消失對都市洪水風險有什麼影響？

> *Your answer:*

---
## S7 — Vegetation Resilience (植被韌性) ✏️ EXERCISE

Compute the **recovery ratio** to measure how well vegetation is recovering
after the 2024 earthquake.

| Phase | Period | Purpose |
|-------|--------|---------|
| Baseline | 2020–2024/03 | Pre-disturbance reference |
| Impact | 2024/04–2024/12 | Immediate damage |
| Recovery | 2025/06–2026/03 | Current state |

**Recovery ratio = (Recovery − Impact) / (Baseline − Impact)**

In [ ]:
# ============================================================
# S7 — Resilience Metrics — EXERCISE
# ============================================================

# TODO: Compute three NDVI composites
# HINT: Same pattern as W13 — filterDate, select NDVI, median
baseline = landsat_idx.filterDate('2020-01-01', '2024-03-31').select('NDVI').median()
impact   = landsat_idx.filterDate('____', '____').select('NDVI').median()
recovery = landsat_idx.filterDate('____', '____').select('NDVI').median()

# Compute recovery ratio
impact_mag = baseline.subtract(impact)
recovery_mag = recovery.subtract(impact)
recovery_ratio = recovery_mag.divide(impact_mag.max(0.01)).clamp(0, 1.5).rename('recovery_ratio')

# Mask: only where impact > 0.1 NDVI
recovery_ratio_masked = recovery_ratio.updateMask(impact_mag.gt(0.1))

# Visualization
vis_rec = {'min': 0, 'max': 1.2,
           'palette': ['8B0000', 'FF4500', 'FFD700', '90EE90', '228B22']}

Map7 = geemap.Map(center=[24.20, 121.60], zoom=11)
Map7.addLayer(recovery_ratio_masked.clip(aoi), vis_rec, 'Recovery Ratio')
Map7.addLayer(aoi, {'color': 'yellow'}, 'AOI')
Map7

### S7 — Resilience Interpretation (韌性解讀)

**Q1:** On the map, which areas show full recovery (green, ratio > 0.8)?
What type of terrain are they on? (e.g., gentle slopes, valleys, ridges)

> *Your answer:*

**Q2:** Which areas show no recovery (red, ratio < 0.3)?
Are these landslide scars, bare rock, or something else?

> *Your answer:*

**Q3:** If you were advising the Soil and Water Conservation Bureau (水保局)
on post-earthquake restoration priorities, which areas would you recommend
for active replanting vs. leaving for natural recovery? Why?

> *Your answer:*

---
## S8b — Landsat × Sentinel-2 Cross-Sensor Analysis（跨感測器分析）✏️ EXERCISE

Landsat 看得遠（26 年），Sentinel-2 看得細（10m）。
現在你要結合兩者：用 Landsat 的長期趨勢找到變遷熱區，
再用 Sentinel-2 的高解析度驗證細節。

### 你需要完成的步驟：
1. 載入 Sentinel-2 資料（跟 W13 一樣的方法）
2. 計算 S2 的地震前後 ΔNDVI
3. 比較 Landsat vs S2 在重疊時段（2017–2026）的 NDVI 差異
4. 分析多解析度差異的原因

In [ ]:
# ============================================================
# S8b — Cross-Sensor Analysis Exercise
# ============================================================

# Step 1: Load Sentinel-2 (same cloud masking as W13)
def mask_s2_clouds(image):
    scl = image.select('SCL')
    mask = scl.gte(4).And(scl.lte(7))
    return image.updateMask(mask)

def add_s2_ndvi(image):
    # TODO: Calculate NDVI using Sentinel-2 bands
    # Hint: S2 NIR = B8, Red = B4
    ndvi = image.normalizedDifference([____, ____]).rename('NDVI_S2')
    return image.addBands(ndvi).copyProperties(image, ['system:time_start'])

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filterDate('2017-04-01', '2026-03-31')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
      .map(mask_s2_clouds)
      .map(add_s2_ndvi))

print(f'Sentinel-2 images loaded: {s2.size().getInfo()}')
print(f'Landsat images (same period): '
      f'{landsat_idx.filterDate("2017-01-01", "2026-03-31").size().getInfo()}')

In [ ]:
# Step 2: Compute earthquake ΔNDVI from BOTH sensors

# TODO: Create Landsat pre/post composites and compute change
l_pre  = landsat_idx.filterDate(____,  ____).select('NDVI').median()
l_post = landsat_idx.filterDate(____,  ____).select('NDVI').median()
l_change = l_post.subtract(l_pre).rename('delta_NDVI_landsat')

# TODO: Create S2 pre/post composites and compute change
s2_pre  = s2.filterDate(____,  ____).select('NDVI_S2').median().rename('NDVI')
s2_post = s2.filterDate(____,  ____).select('NDVI_S2').median().rename('NDVI')
s2_change = s2_post.subtract(s2_pre).rename('delta_NDVI_s2')

# Visualize both on the same map
vis_delta = {'min': -0.3, 'max': 0.1,
             'palette': ['d73027', 'fc8d59', 'fee08b', 'ffffbf',
                         'd9ef8b', '91cf60', '1a9850']}

Map_cs = geemap.Map(center=[24.18, 121.55], zoom=12)
Map_cs.addLayer(l_change.clip(aoi),  vis_delta, 'Landsat ΔNDVI (30m)')
Map_cs.addLayer(s2_change.clip(aoi), vis_delta, 'Sentinel-2 ΔNDVI (10m)')
Map_cs.addLayer(aoi, {'color': 'yellow'}, 'AOI')
Map_cs
# Toggle between the two layers — what differences do you see?

In [ ]:
# Step 3: Annual NDVI cross-sensor comparison (2017–2026)

years_overlap = list(range(2017, 2027))
l_vals, s2_vals = [], []

for yr in years_overlap:
    # Landsat annual mean NDVI
    l_yr = landsat_idx.filterDate(f'{yr}-01-01', f'{yr}-12-31') \
        .select('NDVI').median()
    l_result = l_yr.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi,
        scale=100, maxPixels=1e8
    ).getInfo()
    l_mean = l_result.get('NDVI')

    # TODO: Compute S2 annual mean NDVI (same method)
    s2_yr_col = s2.filterDate(f'{yr}-01-01', f'{yr}-12-31')
    s2_n = s2_yr_col.size().getInfo()
    if s2_n == 0:
        s_mean = None
    else:
        s_yr = s2_yr_col.select(____).median()
        s_result = s_yr.reduceRegion(
            reducer=ee.Reducer.mean(), geometry=aoi,
            scale=100, maxPixels=1e8
        ).getInfo()
        s_mean = s_result.get(____)

    l_vals.append(l_mean)
    s2_vals.append(s_mean)

    l_str = f'{l_mean:.4f}' if l_mean else 'N/A'
    s_str = f'{s_mean:.4f}' if s_mean else 'N/A'
    print(f'  {yr}: Landsat={l_str}  S2={s_str}')

# --- Plot ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(years_overlap, l_vals, 's-', color='#8B4513',
        markersize=8, linewidth=2, label='Landsat (30m)')
ax.plot(years_overlap, s2_vals, 'o-', color='#2E8B57',
        markersize=8, linewidth=2, label='Sentinel-2 (10m)')
ax.axvline(2024, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax.set_xlabel('Year')
ax.set_ylabel('Mean NDVI')
ax.set_title('Cross-Sensor NDVI: Landsat vs Sentinel-2 (2017–2026)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### S8b — Reflection: Cross-Sensor Analysis（跨感測器反思）

**Q1:** 在兩個感測器的 NDVI 時序圖中，Sentinel-2 的值通常比 Landsat 高還是低？
你認為原因是什麼？（提示：想想 30m vs 10m 像素中混合了什麼地物）

> Your answer: ____

**Q2:** 切換 Landsat 和 S2 的 ΔNDVI 圖層，S2 能看到哪些 Landsat 看不到的細節？
（例如：個別崩塌面的形狀？道路中斷的位置？）

> Your answer: ____

**Q3:** 如果你是災害分析師，什麼時候會選擇用 Landsat？什麼時候會選擇用 Sentinel-2？
什麼時候兩者都要用？

> Your answer: ____

**Q4:** 在你的 W15 期末提案中，你會選擇用哪個感測器？還是兩者結合？為什麼？

> Your answer: ____

---
## S8 — Final Reflection (期末反思) ✏️ EXERCISE

**Q1:** Compare the tools and time scales across W13 and W14:

| | W13 | W14 |
|---|---|---|
| Satellite | Sentinel-2 (10m) | Landsat (30m) |
| Time span | 6 years | 26 years |
| Best for | _____ | _____ |
| New technique | _____ | _____ |

> *Fill in the blanks above.*

**Q2:** W6 used Kriging for **spatial** interpolation (points → surface).
W14 uses GEE for **temporal** analysis (snapshots → trends).
How do spatial and temporal perspectives complement each other in
disaster monitoring?

> *Your answer:*

**Q3:** W14 uses two different study areas: Taroko (vegetation/resilience)
and Taoyuan (pond disappearance). Both demonstrate "long-term change
detection" but tell very different stories. What does each case reveal
about human-environment interaction over 26 years?

> *Your answer:*

**Q4:** Looking back at W8–W14, if you could design the ideal monitoring
system for a mountain area like Taroko, which sensors, indices, and time
scales would you combine?

> *Your answer:*

---

## Notebook Complete

```
S1: Setup → S2: Harmonize → S3: Indices → S4: Annual NDVI
→ S5: Trend Map → S6: Pond Disappearance → S7: Resilience → S8: Reflection
```

> *"From snapshots to stories — 從快照到故事"*